In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
RANDOM_STATE = 42
SEED_LIST = [42, 202]     # dipakai utk rata-rata hasil akhir ("seed averaging"). Tambah anggota list ini kalau mau lebih akurat (tapi lebih lama)
N_FOLDS = 5
TUNE_FOLDS = 3
N_TRIALS = 20              # titik tengah: cukup buat cari setelan bagus, tapi tidak terlalu banyak sampai overfit ke validasi
TE_SMOOTH_K = 20            # semakin besar, semakin "hati-hati" percaya rata-rata source yang sample-nya sedikit

In [3]:
DATA_DIR = '/kaggle/input/competitions/seleksi-data-science-academy-compfest-18'
OUT_DIR  = '/kaggle/working/'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

In [4]:
TARGET = 'property_organic_content'
ID_COL = 'sample_id'
SOURCE_COL = 'source_id'

In [5]:
# sampling_depth_cm TIDAK dimasukkan -> isinya sama semua ("0-20"), nol informasi
cat_cols = ['source_id', 'has_band_A_spectrum', 'has_band_B_spectrum',
            'sampling_strategy', 'geo_zone_macro',
            'geo_zone_micro', 'geo_zone_meso', 'land_cover_type',
            'biome', 'parent_rock_type']

In [6]:
def feature_engineer(df):
    df = df.copy()
    # 6 flag original sigma3 -- TERBUKTI paling kuat, tidak ditambah apapun lagi
    df['flag_has_coord'] = df['latitude'].notna().astype(int)
    df['flag_has_acidity'] = df['property_acidity_index'].notna().astype(int)
    df['flag_has_cation_na'] = df['cation_Na'].notna().astype(int)
    df['flag_has_cation_set'] = df['cation_Ca'].notna().astype(int)
    df['flag_has_particle'] = df['property_particle_coarse'].notna().astype(int)
    df['flag_has_band_B'] = (df['has_band_B_spectrum'] == 'YES').astype(int)
    for c in cat_cols:
        df[c] = df[c].astype('category')
    return df

In [7]:
train_fe = feature_engineer(train)
test_fe = feature_engineer(test)

In [8]:
for c in cat_cols:
    cats = pd.api.types.union_categoricals([train_fe[c], test_fe[c]]).categories
    train_fe[c] = train_fe[c].cat.set_categories(cats)
    test_fe[c] = test_fe[c].cat.set_categories(cats)

In [9]:
# XGBoost butuh kategori sebagai angka (bukan teks)
train_xgb = train_fe.copy()
test_xgb = test_fe.copy()
for c in cat_cols:
    train_xgb[c] = train_xgb[c].cat.codes.replace(-1, np.nan)
    test_xgb[c] = test_xgb[c].cat.codes.replace(-1, np.nan)

In [10]:
# CatBoost bisa pakai kategori as-is asal tipe string
train_cb = train_fe.copy()
test_cb = test_fe.copy()
for c in cat_cols:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c] = test_cb[c].astype(str)

In [11]:
drop_cols = [ID_COL, TARGET, 'sampling_depth_cm']
feature_cols = [c for c in train_fe.columns if c not in drop_cols]
cat_col_idx = [feature_cols.index(c) for c in cat_cols]  # dibutuhkan CatBoost

In [12]:
X = train_fe[feature_cols]
y = np.log1p(train_fe[TARGET])
X_test = test_fe[feature_cols]

X_xgb = train_xgb[feature_cols]
X_test_xgb = test_xgb[feature_cols]

X_cb = train_cb[feature_cols]
X_test_cb = test_cb[feature_cols]

source_raw_train = train[SOURCE_COL].astype(str)
source_raw_test = test[SOURCE_COL].astype(str)

## Fungsi Target Encoding untuk `source_id`

Ide sederhananya: tiap sumber (`source_id`) punya "kebiasaan" kadar organik sendiri (ada sumber yang rata-rata rendah ~10, ada yang tinggi ~100). Kita hitung rata-rata itu HANYA dari data latihan (supaya tidak "bocor" info jawaban test), dengan sedikit "kehati-hatian" (`TE_SMOOTH_K`) untuk sumber yang cuma punya sedikit sampel (misal cuma 4 baris) — supaya tidak asal percaya rata-rata dari sampel yang kelewat sedikit.

In [13]:
def compute_source_stats(source_train, y_train, k=TE_SMOOTH_K):
    g_mean = y_train.mean()
    stats = y_train.groupby(source_train).agg(['mean', 'count'])
    smooth = (stats['count'] * stats['mean'] + k * g_mean) / (stats['count'] + k)
    return smooth, g_mean

def apply_source_te(source_series, smooth_map, g_mean):
    return source_series.map(smooth_map).fillna(g_mean).values

In [14]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
tune_kf = KFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=RANDOM_STATE)

## Tuning LightGBM

In [15]:
def lgb_cv_rmse(params):
    oof = np.zeros(len(X))
    for tr_idx, val_idx in tune_kf.split(X):
        src_tr, src_val = source_raw_train.iloc[tr_idx], source_raw_train.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        smooth_map, g_mean = compute_source_stats(src_tr, y_tr)

        X_tr = X.iloc[tr_idx].copy(); X_tr['source_te'] = apply_source_te(src_tr, smooth_map, g_mean)
        X_val = X.iloc[val_idx].copy(); X_val['source_te'] = apply_source_te(src_val, smooth_map, g_mean)

        dtr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=dtr)
        model = lgb.train(params, dtr, num_boost_round=2000, valid_sets=[dval],
                           callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    return mean_squared_error(y, oof) ** 0.5

In [16]:
def lgb_objective(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'seed': RANDOM_STATE,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 60),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': 1,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
    }
    return lgb_cv_rmse(params)

In [17]:
print("Tuning LightGBM...")
lgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print("Best LGB CV RMSE:", lgb_study.best_value)
print("Best LGB params:", lgb_study.best_params)

Tuning LightGBM...
Best LGB CV RMSE: 0.29071350897017423
Best LGB params: {'learning_rate': 0.010507993830373511, 'num_leaves': 103, 'min_data_in_leaf': 55, 'feature_fraction': 0.8364610680219806, 'bagging_fraction': 0.6135020205326719, 'lambda_l1': 0.5197473211490475, 'lambda_l2': 0.5517443901266632}


## Tuning XGBoost

In [18]:
def xgb_cv_rmse(params):
    oof = np.zeros(len(X_xgb))
    for tr_idx, val_idx in tune_kf.split(X_xgb):
        src_tr, src_val = source_raw_train.iloc[tr_idx], source_raw_train.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        smooth_map, g_mean = compute_source_stats(src_tr, y_tr)

        X_tr = X_xgb.iloc[tr_idx].copy(); X_tr['source_te'] = apply_source_te(src_tr, smooth_map, g_mean)
        X_val = X_xgb.iloc[val_idx].copy(); X_val['source_te'] = apply_source_te(src_val, smooth_map, g_mean)

        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof[val_idx] = model.predict(X_val)
    return mean_squared_error(y, oof) ** 0.5

def xgb_objective(trial):
    params = {
        'objective': 'reg:squarederror', 'tree_method': 'hist', 'enable_categorical': False,
        'random_state': RANDOM_STATE, 'n_estimators': 1500, 'early_stopping_rounds': 50,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'n_jobs': -1, 'verbosity': 0,
    }
    return xgb_cv_rmse(params)

In [19]:
print("Tuning XGBoost...")
xgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print("Best XGB CV RMSE:", xgb_study.best_value)
print("Best XGB params:", xgb_study.best_params)

Tuning XGBoost...
Best XGB CV RMSE: 0.29359824765380704
Best XGB params: {'learning_rate': 0.010507993830373511, 'max_depth': 9, 'min_child_weight': 19, 'subsample': 0.8364610680219806, 'colsample_bytree': 0.6079180203564428, 'reg_alpha': 0.07464186719645012, 'reg_lambda': 0.5517443901266632}


## Tuning CatBoost

In [20]:
def cb_cv_rmse(params):
    oof = np.zeros(len(X_cb))
    for tr_idx, val_idx in tune_kf.split(X_cb):
        src_tr, src_val = source_raw_train.iloc[tr_idx], source_raw_train.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        smooth_map, g_mean = compute_source_stats(src_tr, y_tr)

        X_tr = X_cb.iloc[tr_idx].copy(); X_tr['source_te'] = apply_source_te(src_tr, smooth_map, g_mean)
        X_val = X_cb.iloc[val_idx].copy(); X_val['source_te'] = apply_source_te(src_val, smooth_map, g_mean)

        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_col_idx,
                  use_best_model=True, verbose=False)
        oof[val_idx] = model.predict(X_val)
    return mean_squared_error(y, oof) ** 0.5

def cb_objective(trial):
    params = {
        'loss_function': 'RMSE', 'random_seed': RANDOM_STATE, 'iterations': 1500,
        'early_stopping_rounds': 100, 'verbose': False, 'allow_writing_files': False,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
    return cb_cv_rmse(params)

In [21]:
print("Tuning CatBoost...")
cb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
cb_study.optimize(cb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print("Best CatBoost CV RMSE:", cb_study.best_value)
print("Best CatBoost params:", cb_study.best_params)

Tuning CatBoost...
Best CatBoost CV RMSE: 0.2944741607859925
Best CatBoost params: {'learning_rate': 0.04397398346218303, 'depth': 8, 'l2_leaf_reg': 0.018674590124935497, 'bagging_temperature': 0.74363678460855}


In [22]:
best_lgb_params = {'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'seed': RANDOM_STATE}
best_lgb_params.update(lgb_study.best_params)
best_lgb_params['bagging_freq'] = 1

best_xgb_params = {'objective': 'reg:squarederror', 'tree_method': 'hist', 'enable_categorical': False,
                    'random_state': RANDOM_STATE, 'n_estimators': 3000, 'early_stopping_rounds': 100,
                    'n_jobs': -1, 'verbosity': 0}
best_xgb_params.update(xgb_study.best_params)

best_cb_params = {'loss_function': 'RMSE', 'random_seed': RANDOM_STATE, 'iterations': 3000,
                   'early_stopping_rounds': 150, 'verbose': False, 'allow_writing_files': False}
best_cb_params.update(cb_study.best_params)

## Training Akhir — Dirata-rata dari Beberapa Seed (Seed Averaging)

Kenapa perlu ini? Pembagian data ke 5 fold itu sedikit "acak" (random). Dengan cuma 1 kali pembagian (1 seed), hasil akhir bisa sedikit terpengaruh "keberuntungan" cara data dibagi. Dengan mengulang seluruh proses training pakai 2 cara pembagian berbeda (2 seed) lalu dirata-rata, hasil akhirnya jadi lebih stabil — bukan menambah info baru, cuma mengurangi pengaruh faktor acak.

In [23]:
oof_lgb = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
oof_xgb = np.zeros(len(X_xgb)); test_xgb_pred = np.zeros(len(X_test_xgb))
oof_cb = np.zeros(len(X_cb)); test_cb_pred = np.zeros(len(X_test_cb))

In [24]:
n_seeds = len(SEED_LIST)

for seed_num, seed in enumerate(SEED_LIST):
    print(f"\n===== Seed {seed_num+1}/{n_seeds} (seed={seed}) =====")
    kf_seed = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold_num, (tr_idx, val_idx) in enumerate(kf_seed.split(X)):
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        src_tr, src_val = source_raw_train.iloc[tr_idx], source_raw_train.iloc[val_idx]
        smooth_map, g_mean = compute_source_stats(src_tr, y_tr)
        te_test = apply_source_te(source_raw_test, smooth_map, g_mean)

        # ---- LightGBM ----
        X_tr = X.iloc[tr_idx].copy(); X_tr['source_te'] = apply_source_te(src_tr, smooth_map, g_mean)
        X_val = X.iloc[val_idx].copy(); X_val['source_te'] = apply_source_te(src_val, smooth_map, g_mean)
        X_test_te = X_test.copy(); X_test_te['source_te'] = te_test

        dtr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=dtr)
        m_lgb = lgb.train(best_lgb_params, dtr, num_boost_round=3000, valid_sets=[dval],
                           callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof_lgb[val_idx] += m_lgb.predict(X_val, num_iteration=m_lgb.best_iteration) / n_seeds
        test_lgb += m_lgb.predict(X_test_te, num_iteration=m_lgb.best_iteration) / (N_FOLDS * n_seeds)

        # ---- XGBoost ----
        X_tr_x = X_xgb.iloc[tr_idx].copy(); X_tr_x['source_te'] = apply_source_te(src_tr, smooth_map, g_mean)
        X_val_x = X_xgb.iloc[val_idx].copy(); X_val_x['source_te'] = apply_source_te(src_val, smooth_map, g_mean)
        X_test_x_te = X_test_xgb.copy(); X_test_x_te['source_te'] = te_test

        m_xgb = xgb.XGBRegressor(**best_xgb_params)
        m_xgb.fit(X_tr_x, y_tr, eval_set=[(X_val_x, y_val)], verbose=False)
        oof_xgb[val_idx] += m_xgb.predict(X_val_x) / n_seeds
        test_xgb_pred += m_xgb.predict(X_test_x_te) / (N_FOLDS * n_seeds)

        # ---- CatBoost ----
        X_tr_c = X_cb.iloc[tr_idx].copy(); X_tr_c['source_te'] = apply_source_te(src_tr, smooth_map, g_mean)
        X_val_c = X_cb.iloc[val_idx].copy(); X_val_c['source_te'] = apply_source_te(src_val, smooth_map, g_mean)
        X_test_c_te = X_test_cb.copy(); X_test_c_te['source_te'] = te_test

        m_cb = CatBoostRegressor(**best_cb_params)
        m_cb.fit(X_tr_c, y_tr, eval_set=(X_val_c, y_val), cat_features=cat_col_idx,
                 use_best_model=True, verbose=False)
        oof_cb[val_idx] += m_cb.predict(X_val_c) / n_seeds
        test_cb_pred += m_cb.predict(X_test_c_te) / (N_FOLDS * n_seeds)

        print(f"  Fold {fold_num+1}/{N_FOLDS} selesai")


===== Seed 1/2 (seed=42) =====
  Fold 1/5 selesai
  Fold 2/5 selesai
  Fold 3/5 selesai
  Fold 4/5 selesai
  Fold 5/5 selesai

===== Seed 2/2 (seed=202) =====
  Fold 1/5 selesai
  Fold 2/5 selesai
  Fold 3/5 selesai
  Fold 4/5 selesai
  Fold 5/5 selesai


In [25]:
rmse_lgb = mean_squared_error(y, oof_lgb) ** 0.5
rmse_xgb = mean_squared_error(y, oof_xgb) ** 0.5
rmse_cb = mean_squared_error(y, oof_cb) ** 0.5
print(f"\nFinal OOF RMSE (skala log) - LightGBM: {rmse_lgb:.5f}")
print(f"Final OOF RMSE (skala log) - XGBoost : {rmse_xgb:.5f}")
print(f"Final OOF RMSE (skala log) - CatBoost: {rmse_cb:.5f}")

rmse_lgb_raw = mean_squared_error(np.expm1(y), np.expm1(oof_lgb)) ** 0.5
rmse_xgb_raw = mean_squared_error(np.expm1(y), np.expm1(oof_xgb)) ** 0.5
rmse_cb_raw = mean_squared_error(np.expm1(y), np.expm1(oof_cb)) ** 0.5
print(f"\nFinal OOF RMSE (skala asli, sepadan leaderboard) - LightGBM: {rmse_lgb_raw:.4f}")
print(f"Final OOF RMSE (skala asli, sepadan leaderboard) - XGBoost : {rmse_xgb_raw:.4f}")
print(f"Final OOF RMSE (skala asli, sepadan leaderboard) - CatBoost: {rmse_cb_raw:.4f}")


Final OOF RMSE (skala log) - LightGBM: 0.28563
Final OOF RMSE (skala log) - XGBoost : 0.28796
Final OOF RMSE (skala log) - CatBoost: 0.28926

Final OOF RMSE (skala asli, sepadan leaderboard) - LightGBM: 11.5488
Final OOF RMSE (skala asli, sepadan leaderboard) - XGBoost : 11.6253
Final OOF RMSE (skala asli, sepadan leaderboard) - CatBoost: 11.7408


## Blending 3 Model

In [26]:
best_weights, best_rmse = (1/3, 1/3, 1/3), 1e9
step = 0.05
for w_lgb in np.arange(0, 1.01, step):
    for w_xgb in np.arange(0, 1.01 - w_lgb, step):
        w_cb = 1 - w_lgb - w_xgb
        blend_oof = w_lgb * oof_lgb + w_xgb * oof_xgb + w_cb * oof_cb
        r = mean_squared_error(np.expm1(y), np.expm1(blend_oof)) ** 0.5
        if r < best_rmse:
            best_rmse = r
            best_weights = (w_lgb, w_xgb, w_cb)

In [27]:
w_lgb, w_xgb, w_cb = best_weights
print(f"Best blend (LGB={w_lgb:.2f}, XGB={w_xgb:.2f}, CAT={w_cb:.2f}) -> OOF RMSE skala asli: {best_rmse:.4f}")

Best blend (LGB=0.65, XGB=0.15, CAT=0.20) -> OOF RMSE skala asli: 11.5181


In [28]:
test_blend = w_lgb * test_lgb + w_xgb * test_xgb_pred + w_cb * test_cb_pred
test_blend = np.clip(np.expm1(test_blend), a_min=0, a_max=None)

submission = pd.DataFrame({ID_COL: test_fe[ID_COL], TARGET: test_blend})
assert len(submission) == 2670
assert submission[TARGET].isna().sum() == 0

submission.to_csv('submission-sigma5.csv', index=False)
print("Selesai! File tersimpan sebagai submission-sigma5.csv")
print(submission.describe())

Selesai! File tersimpan sebagai submission-sigma5.csv
       property_organic_content
count               2670.000000
mean                  33.108081
std                   19.475371
min                    6.095469
25%                   19.476326
50%                   27.128085
75%                   41.696974
max                  162.625477
